In [9]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [10]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a fitted CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)
    
    for node in bn.nodes:
        cpd = bn.cpds[node]
        num_node_states = cpd.shape[0]
        num_parent_configs = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)
        parents = list(bn.structure.predecessors(node))
        
        if len(parents) == 0:
            probs = data[node].map(cpd.iloc[:, 0]).values
        else:
            cpd_series = cpd.unstack()
            parent_order = sorted(parents)
            lookup_cols = parent_order + [node]
            lookup_df = data[lookup_cols]
            lookup_keys = [tuple(x) for x in lookup_df.to_numpy()]
            probs = cpd_series.loc[lookup_keys].values
            
        log_likelihood += np.sum(np.log(probs + 1e-9))
    
    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [11]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [12]:
def discretize_data(df):
    discretised_df = pd.DataFrame(index=df.index)
    TARGET_BINS = 3

    for col in df.columns:
        if df[col].isna().all():
            discretised_df[col] = "Missing"
            continue
        
        no_unique = df[col].nunique()
    
        if no_unique <= 1:
            discretised_df[col] = "Constant"
            continue
    
        if no_unique <= TARGET_BINS:
            discretised_df[col] = df[col].rank(method='dense') - 1
        else:
            try:
                discretised_df[col] = pd.qcut(df[col], q=TARGET_BINS, labels=False, duplicates='drop')
            except (ValueError, TypeError):
                try:
                    discretised_df[col] = pd.cut(df[col], bins=TARGET_BINS, labels=False, include_lowest=True)
                except (ValueError, TypeError):
                    discretised_df[col] = pd.qcut(df[col].rank(method='first'), q=TARGET_BINS, labels=False, duplicates='drop')

        if pd.api.types.is_numeric_dtype(discretised_df[col]):
            valid_values = sorted(discretised_df[col].dropna().unique())
            num_states = len(valid_values)
            
            if num_states == 2:
                label_map = {valid_values[0]: "Low", valid_values[1]: "High"}
            elif num_states == 3:
                label_map = {valid_values[0]: "Low", valid_values[1]: "Medium", valid_values[2]: "High"}
            elif num_states == 4:
                label_map = {valid_values[0]: "VeryLow", valid_values[1]: "Low", valid_values[2]: "High", valid_values[3]: "VeryHigh"}
            else:
                label_map = {val: f"State{int(val)}" for val in valid_values}
            
            discretised_df[col] = discretised_df[col].map(label_map)
            
            discretised_df[col] = discretised_df[col].fillna("Unknown")

    print("Data types after discretization:")
    print(discretised_df.dtypes.value_counts())

    print("\nUnique states per column:")
    print(discretised_df.nunique())

    print("\nDiscretised data preview:")
    print(discretised_df.head())

    if discretised_df.isna().any().any():
        print("\n⚠️ Warning: Some NaN values remain!")
        print(discretised_df.isna().sum()[discretised_df.isna().sum() > 0])
    
    return discretised_df

In [13]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

df_de = df_de.drop(["Unnamed: 0"], axis = 1)

df_de

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,35330.0,26.46,77.264286,6.710000,6.550000,11.82,10.039,6.389770,17.555365,-0.168868,1.46,-1.2,15.2,0.000010,581.657956,6.196333
1,35750.0,26.06,77.264286,6.670000,6.540000,10.93,10.072,5.783688,17.951098,-0.295519,0.81,-2.6,15.2,0.000010,599.683352,10.012677
2,33880.0,23.87,73.200000,6.610000,6.490000,11.31,10.851,4.768184,19.597935,-2.070729,1.63,-2.4,15.5,0.000009,584.358974,37.999363
3,35370.0,26.06,74.000000,6.610000,6.420000,10.42,11.667,5.103897,19.176892,-3.199506,2.48,-0.6,15.6,0.000010,575.287356,34.793707
4,36700.0,26.31,75.400000,6.450000,6.150000,8.93,12.470,4.852664,18.693432,-0.370659,2.55,1.3,15.8,0.000010,585.349446,39.857009
5,36820.0,26.58,75.800000,6.350000,6.090000,8.12,13.549,6.037145,18.897792,0.092987,1.82,2.4,16.1,0.000010,579.634465,40.920931
6,36910.0,25.91,76.300000,6.080000,6.010000,7.82,13.757,5.772928,19.247998,0.204270,1.76,3.0,16.1,0.000010,587.141971,52.698567
7,37600.0,26.01,76.700000,5.940000,5.880000,7.61,14.381,6.537450,19.210129,0.684115,0.44,5.3,16.7,0.000010,594.640652,39.295143
8,37950.0,25.92,77.000000,5.950000,5.720000,7.33,14.901,7.417530,19.312236,0.679371,1.82,12.0,16.7,0.000010,580.046517,27.981817
9,38550.0,26.22,77.600000,5.900000,5.600000,7.08,14.885,7.263048,19.519072,0.736708,1.64,4.2,16.5,0.000010,588.706740,8.995473


In [14]:
de = pd.DataFrame()
de = df_de.copy()
de['EUPC_3'] = de['EUPC'].shift(3)

In [15]:
de = de.dropna()

In [16]:
discretised_de = discretize_data(de)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_3    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP     QOA     QOR     YUA REST    EBSG GGFC    LBGD     FDI  \
3  Low  Medium  Low    High    High    High  Low     Low  Low     Low  Medium   
4  Low    High  Low    High    High    High  Low     Low  Low  Medium  Medium   
5  Low    High  Low    High    High    High  Low  Medium  Low  Medium     Low   
6  Low     Low  Low  Medium    High  Medium  Low  Medium  Low  Medium     Low   
7  Low  Medium  Low     Low  Medium  Medium  Low    High  Low    High     Low   

       PC     ARP    CO2C     PAM    EUPC  EUPC_3  
3     Low     Low    High  Medium  Medium     Low  
4  Medium     Low    High    High    High     Low  
5  Medium  Med

In [17]:
sm = StructureModel()

In [18]:
sm.add_edges_from([
    ('EUPC_3', 'ARP')
])

In [19]:
sm.edges

OutEdgeView([('EUPC_3', 'ARP')])

In [20]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [22]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_de)
bn_baseline.fit_cpds(discretised_de, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_de)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# 3. TEST #1: (EUPC_3  -> ARP)
# ==========================================
print("\n--- Testing EUPC_3 -> ARP ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_3', 'ARP'], 
    discretised_data=discretised_de,
    raw_data=de
)

BASELINE BIC: 68.75

--- Testing EUPC_3 -> ARP ---
Estimating LV based on continuous GMM distributions in: ['EUPC_3', 'ARP']
  - Removing direct edge: EUPC_3 -> ARP
LV Model BIC: 87.64 (LogLikelihood: -32.28)


In [45]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_bn_no_context.csv")

df_de = df_de.drop(["Unnamed: 0"], axis = 1)

df_de

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,35330.0,26.46,77.264286,6.710000,6.550000,11.82,10.039,6.389770,17.555365,-0.168868,1.46,-1.2,15.2,0.000010,581.657956,6.196333
1,35750.0,26.06,77.264286,6.670000,6.540000,10.93,10.072,5.783688,17.951098,-0.295519,0.81,-2.6,15.2,0.000010,599.683352,10.012677
2,33880.0,23.87,73.200000,6.610000,6.490000,11.31,10.851,4.768184,19.597935,-2.070729,1.63,-2.4,15.5,0.000009,584.358974,37.999363
3,35370.0,26.06,74.000000,6.610000,6.420000,10.42,11.667,5.103897,19.176892,-3.199506,2.48,-0.6,15.6,0.000010,575.287356,34.793707
4,36700.0,26.31,75.400000,6.450000,6.150000,8.93,12.470,4.852664,18.693432,-0.370659,2.55,1.3,15.8,0.000010,585.349446,39.857009
5,36820.0,26.58,75.800000,6.350000,6.090000,8.12,13.549,6.037145,18.897792,0.092987,1.82,2.4,16.1,0.000010,579.634465,40.920931
6,36910.0,25.91,76.300000,6.080000,6.010000,7.82,13.757,5.772928,19.247998,0.204270,1.76,3.0,16.1,0.000010,587.141971,52.698567
7,37600.0,26.01,76.700000,5.940000,5.880000,7.61,14.381,6.537450,19.210129,0.684115,0.44,5.3,16.7,0.000010,594.640652,39.295143
8,37950.0,25.92,77.000000,5.950000,5.720000,7.33,14.901,7.417530,19.312236,0.679371,1.82,12.0,16.7,0.000010,580.046517,27.981817
9,38550.0,26.22,77.600000,5.900000,5.600000,7.08,14.885,7.263048,19.519072,0.736708,1.64,4.2,16.5,0.000010,588.706740,8.995473


In [46]:
de = pd.DataFrame()
de = df_de.copy()
de['FDI_2'] = de['FDI'].shift(2)
de['GGFC_2'] = de['GGFC'].shift(2)
de['PC_1'] = de['PC'].shift(1)

In [47]:
de = de.dropna()

In [48]:
discretised_de = discretize_data(de)

Data types after discretization:
object    19
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
FDI_2     3
GGFC_2    3
PC_1      3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP     QOA   QOR     YUA REST    EBSG  GGFC    LBGD     FDI  \
2  Low     Low  Low    High  High    High  Low     Low  High     Low     Low   
3  Low    High  Low    High  High    High  Low     Low   Low     Low  Medium   
4  Low    High  Low    High  High    High  Low     Low   Low  Medium    High   
5  Low    High  Low    High  High    High  Low    High   Low  Medium     Low   
6  Low  Medium  Low  Medium  High  Medium  Low  Medium   Low  Medium     Low   

       PC     ARP    CO2C     PAM    EUPC FDI_2 GGFC_2    PC_1  
2     Low     Low  Medium    High    High   Low    Low     Low  
3     Low     Low    H

In [49]:
sm = StructureModel()

In [54]:
sm.add_edges_from([
    ('EUPC', 'VABI'),
    ('EUPC', 'EBSG'),
    ('QOA', 'EUPC'),
    ('FDI_2', 'EUPC'),
     ('PC_1', 'EUPC'),
    ('GGFC_2', 'EUPC'),
    
])

In [55]:
sm.edges

OutEdgeView([('PC_1', 'EUPC'), ('EUPC', 'VABI'), ('EUPC', 'EBSG'), ('GGFC_2', 'EUPC'), ('QOA', 'EUPC'), ('FDI_2', 'EUPC')])

In [56]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [57]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_de)
bn_baseline.fit_cpds(discretised_de, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_de)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> VABI)
# ==========================================
print("\n--- Testing EUPC -> VABI ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'VABI'], 
    discretised_data=discretised_de,
    raw_data=de
)

# ==========================================
# TEST #2: (EUPC -> EBSG)
# ==========================================
print("\n--- Testing EUPC -> EBSG ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['EUPC', 'EBSG'], 
    discretised_data=discretised_de,
    raw_data=de
)

# ==========================================
# TEST #3: (QOA -> EUPC)
# ==========================================
print("\n--- Testing QOA -> EUPC ---")

sm_test_3 = sm_baseline.copy()

bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_3,
    children_cols=['QOA', 'EUPC'], 
    discretised_data=discretised_de,
    raw_data=de
)

# ==========================================
# TEST #4: (FDI_2 -> EUPC)
# ==========================================
print("\n--- Testing FDI_2 -> EUPC ---")

sm_test_4 = sm_baseline.copy()

bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_4,
    children_cols=['FDI_2', 'EUPC'], 
    discretised_data=discretised_de,
    raw_data=de
)
# ==========================================
# TEST #5: (PC_1 -> EUPC)
# ==========================================
print("\n--- Testing PC_1 -> EUPC ---")

sm_test_5 = sm_baseline.copy()

bn_with_lv_5, data_with_lv_5, bic_lv_5 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_5,
    children_cols=['PC_1', 'EUPC'], 
    discretised_data=discretised_de,
    raw_data=de
)
# ==========================================
# 3.TEST #6: (GGFC_2 -> EUPC)
# ==========================================
print("\n--- Testing GGFC_2  -> EUPC ---")

sm_test_6 = sm_baseline.copy()

bn_with_lv_6, data_with_lv_6, bic_lv_6 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_6,
    children_cols=['GGFC_2', 'EUPC'], 
    discretised_data=discretised_de,
    raw_data=de
)



BASELINE BIC: 674.75

--- Testing EUPC -> VABI ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'VABI']
  - Removing direct edge: EUPC -> VABI
LV Model BIC: 1109.08 (LogLikelihood: -101.94)

--- Testing EUPC -> EBSG ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'EBSG']
  - Removing direct edge: EUPC -> EBSG
LV Model BIC: 1112.51 (LogLikelihood: -103.66)

--- Testing QOA -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['QOA', 'EUPC']
  - Removing direct edge: QOA -> EUPC
LV Model BIC: 547.42 (LogLikelihood: -100.85)

--- Testing FDI_2 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['FDI_2', 'EUPC']
  - Removing direct edge: FDI_2 -> EUPC
LV Model BIC: 546.25 (LogLikelihood: -100.27)

--- Testing PC_1 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['PC_1', 'EUPC']
  - Removing direct edge: PC_1 -> EUPC
LV Model BIC: 547.87 (LogLikelihood: -101.08)

--- Testing GGFC_2  -> EUPC ---
Esti